In [1]:
import json
from pathlib import Path

import stable_whisper
from langchain_text_splitters import RecursiveCharacterTextSplitter
import ollama
from tqdm import tqdm
from llama_cpp import Llama
import yt_dlp

In [2]:
def download_audio(url: str, save_dir: Path, save_name: str) -> str:
    output_path = Path(f"{save_dir}/{save_name}.mp3")
    with yt_dlp.YoutubeDL({'extract_audio': True, 'format': 'bestaudio', 'outtmpl': f'{output_path}'}) as video:
        video.download(url)
    return str(output_path)
    
def postprocess_chunk(chunk: str) -> str:
    chunk = chunk.lstrip(". ")
    chunk = chunk.capitalize()
    chunk += "."
    return chunk

def ollama_chat(model_name: str, prompt: str, format: str = "") -> str:
    response = ollama.chat(model=model_name,
                           messages=[
                                        {
                                            'role': 'user',
                                            'content': prompt,
                                        },
                                    ],
                           stream=False,
                           format=format)
    return response["message"]["content"]

def load_llm(gguf_path: str) -> Llama:
    llm = Llama(model_path=gguf_path, chat_format="chatml", verbose=False)
    return llm

def llamacpp_chat(llm, system_prompt: str, user_prompt: str, schema: dict) -> str:
    response = llm.create_chat_completion(
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {"role": "user", "content": user_prompt},
        ],
        response_format={
            "type": "json_object",
            "schema": schema,
        },
        temperature=0.0,
    )
    return response["choices"][0]["message"]["content"]

def summarize_chunk(chunk: str) -> str:
    return ollama_chat("llama3_instruct_podcast_chunk_summarizer", chunk)

BULLETIZATION_SYSTEM_PROMPT = """You are a helpful assistant that outputs in JSON. You are an expert at converting text into short points. Convert the given text into short points and a title. Don't add escape characters. Directly list the points and the title, don't add additional text before or after it."""
BULLETIZATION_SCHEMA = {
                            "type": "object",
                            "properties": {"title": {"type": "string"},
                                            "bullets": {"type": "array", "items": {"type": "string"}}
                                            },
                            "required": ["title", "bullets"],
                        }

def bulletize_summary(summary: str, llm: Llama) -> str:
    response_text = llamacpp_chat(llm, 
                                  system_prompt=BULLETIZATION_SYSTEM_PROMPT, 
                                  user_prompt=summary,
                                  schema=BULLETIZATION_SCHEMA)
    return json.loads(response_text)

In [3]:
TEST_FILE_PATH = download_audio(url="https://www.youtube.com/watch?v=0m3hGZvD-0s",
                                save_dir="/home/jobin/Projects/multimodal_rag_summarization/data",
                                save_name="test2")

[youtube] Extracting URL: https://www.youtube.com/watch?v=0m3hGZvD-0s
[youtube] 0m3hGZvD-0s: Downloading webpage
[youtube] 0m3hGZvD-0s: Downloading ios player API JSON
[youtube] 0m3hGZvD-0s: Downloading android player API JSON


[youtube] 0m3hGZvD-0s: Downloading m3u8 information
[info] 0m3hGZvD-0s: Downloading 1 format(s): 251
[download] /home/jobin/Projects/multimodal_rag_summarization/data/test2.mp3 has already been downloaded
[download] 100% of   26.76MiB


In [4]:
tiny_whisper = stable_whisper.load_model("tiny")
result = tiny_whisper.transcribe(TEST_FILE_PATH, word_timestamps=True)
result_dict = result.to_dict()
transcript = result_dict["text"].strip()

Transcribe:   0%|          | 0/2142.76 [00:00<?, ?sec/s]

Detected language: english


Transcribe: 100%|█████████▉| 2142.75/2142.76 [00:39<00:00, 54.06sec/s]


In [5]:
splitter = RecursiveCharacterTextSplitter(separators=["."],
                                          chunk_size=5000,
                                          chunk_overlap=0,
                                          length_function=len,
                                          is_separator_regex=False)

chunks = splitter.split_text(transcript)
chunks = [postprocess_chunk(x) for x in chunks]

In [5]:
llm = load_llm(gguf_path="/home/jobin/Projects/multimodal_rag_summarization/gguf/Meta-Llama-3-8B-Instruct.Q6_K.gguf")
summary_json_list = []
for chunk in tqdm(chunks):
    summary = summarize_chunk(chunk)
    summary_json = bulletize_summary(summary, llm)
    summary_json_list.append(summary_json)

100%|██████████| 7/7 [02:36<00:00, 22.34s/it]


In [6]:
summary_json_list

[{'title': 'Daily Routine for Maximizing Productivity',
  'bullets': ['Wake up at 6:30 am',
   'Morning mantra includes setting rules for social media use, diet, and exercise',
   "Mantra also involves gratitude, visualizing goals, and focusing on the day's tasks"]},
 {'title': 'Daily Routine',
  'bullets': ['Morning mantra sets the tone for the day',
   'Four-hour focused session of deep work with minimal interruptions',
   'Standing desk and ergonomic keyboard setup',
   'Love for music, particularly guitar playing',
   'Long run at the end of the day with brown noise and audiobook',
   'Reflecting on thoughts and emotions during the run']},
 {'title': 'Daily Routine and Reflections',
  'bullets': ['Running and bodyweight exercises while fasting helps me focus mentally and physically.',
   'I enjoy exercising fasted as it allows me to tap into my mental and physical toughness.',
   'Reflecting on history, particularly Nazi Germany, makes me realize the importance of mental toughness 